In [1]:
.libPaths(c("/home/mattia/miniconda3/envs/Signac/lib/R/library","/home/mattia/R/x86_64-pc-linux-gnu-library/4.2"))

In [2]:
library(tidyverse)
library(IRanges)
library(dplyr)
library(plyranges)
library(ChIPpeakAnno)
library(tibble)
library(readr)
library(biomaRt)
library(plyranges)
library(parallel)
library(ggthemes)
library(viridis)
library(MASS)
library(ashr)
library(LSD)
library(plotfunctions)
library(colorRamps)
setwd("/date/gcb/gcb_MZ/Round1/")

Warning message:
“package ‘purrr’ was built under R version 4.3.1”
Warning message:
“package ‘dplyr’ was built under R version 4.3.1”
Warning message:
“package ‘lubridate’ was built under R version 4.3.1”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.3     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The f

In [ ]:
#Obtain TSS coordinates for all human genome:

TSS <- TSS.human.GRCh38 %>% 
  as.data.frame() %>% 
  rownames_to_column() %>% 
  dplyr::rename(Geneid=1,chr=2) 


# Define the prefix you want to add
prefix <- "chr"

# Add the prefix to the chromosome names using subseqnames
TSS$chr <- paste0(prefix, TSS$chr)

# Print the modified GRanges object
TSS


Geneid <- TSS$Geneid

ensembl <- useEnsembl(biomart = "ensembl", dataset = "hsapiens_gene_ensembl")

listAttributes(ensembl)


TSS_symbol <- getBM(attributes = c("external_gene_name","ensembl_gene_id"),filters = "ensembl_gene_id",values = Geneid,mart = ensembl) %>% 
  dplyr::rename(SYMBOL=1,
                Geneid=2)

TSS <- inner_join(TSS,TSS_symbol) %>% 
  dplyr::rename(chr=2,start=3,end=4) %>% 
  makeGRangesFromDataFrame(keep.extra.columns = T) %>% 
  as.data.frame()%>% 
  dplyr::rename(chr=1,start=2,end=3, strand=5)%>% 
  write_delim("TSS_hg38_list.bed",
              delim="\t",col_names = F)

TSS

Here we calculate counts in Hg38 for H3K27ac samples Paried end 

In [ ]:
#Upload the count matrix generated with deptools

myCounts<- read_delim("/date/gcb/gcb_MZ/Round1/deeptools/summary/multiBamSummary_TSS_K27ac.tab",
delim="\t", col_names=T)%>% 
dplyr::rename(chr=1,start=2,end=3)
  #group_by(chr, start, end) %>%
  #summarise_all(~mean(as.numeric(.))) %>%
  #ungroup()

# Remove the ".bam" suffix from all column names
new_colnames <- sub(".bam", " ", colnames(myCounts))

# Assign the new column names to the data frame
colnames(myCounts) <- new_colnames
colnames(myCounts) <- gsub("'", "", colnames(myCounts))
Norm_factor<-read_tsv("/date/gcb/gcb_MZ/Round1/deeptools/summary/norm_factor_TSS_K27ac.tab")

# Remove ".bam" from the "sample" column in df1
Norm_factor$sample <- sub(".bam$", "", Norm_factor$sample)
Norm_factor$scalingFactor <- as.numeric(Norm_factor$scalingFactor)
# Loop through the sample names in df1 and divide the corresponding column in df2

# Loop through each sample in Norm_factor
for (sample_name in Norm_factor$sample) {
  # Finding the matching index in myCounts
  index <- grep(sample_name, colnames(myCounts))
  
  if (length(index) > 0) {
    # Divide the corresponding column by the scaling factor
    myCounts[, index] <- myCounts[, index] / Norm_factor$scalingFactor[which(Norm_factor$sample == sample_name)]
  } else {
    print(paste("Column", sample_name, "not found in myCounts"))
  }
}


myCounts <- dplyr::select(myCounts, -chr, -start, -end) 
# View the updated df2


In [ ]:
colnames(myCounts) <- gsub("-", "_", colnames(myCounts))
myCounts<-as.data.frame(myCounts)

In [ ]:
library(LSD)
library(viridis)

# Assuming myCounts is your dataset and it has 30 columns
for (i in 1:30) {
  for (j in (i+1):30) {
    filename <- paste0(colnames(myCounts)[i], "_vs_", colnames(myCounts)[j], ".png")
    png(filename = filename, width =35, height = 20, res = 720, units = "cm")
    plot_title <- paste("Comparison of", colnames(myCounts)[i], "vs", colnames(myCounts)[j])
    correlation <- cor(myCounts[, i], myCounts[, j])

    p<-heatscatter(
      x = log2(myCounts[, i]),
      y = log2(myCounts[, j]),
      cexplot = 0.5,
      colpal="bl2gr2rd",
      main = plot_title,
      nrcol = 90,
      grid = 1000,
      add.contour = FALSE,
      xlab = paste("log2 (normalized counts)", colnames(myCounts)[i]),
      ylab = paste("log2 (normalized counts)", colnames(myCounts)[j]),
      xaxs = "i",
      yaxs = "i",
      xlim = c(-3, 20), 
      ylim = c(-3, 20)
    )

    legend("topright", legend = paste("Correlation:", round(correlation, 2)), bty = "n")
   print(p)
    dev.off()
  }
}

In [ ]:
# Create a ggplot with correlation text and point density coloring
ggplot(myCounts, aes(x = myCounts[, 29], y = myCounts[, 30])) +
stat_density_2d(aes(fill = ..level..), geom = "polygon")+
  geom_hex(bins = 70) +
  scale_fill_continuous(type = "viridis") +
  theme_bw()



In [ ]:
pca<- read_tsv("/date/gcb/gcb_MZ/Round1/deeptools/correlation/pca_TSS_K27ac.tab", col_names=T)
# Assuming your initial dataframe is named 'your_data'
pca <- t(pca)
# Assign the values from the first row as colnames
colnames(pca) <- as.character(pca[1,])

# Remove the row that was used to set colnames (if needed)
pca<- pca[-1,]

# Adding the prefix "PC" to the column names in the transposed dataframe
colnames(pca) <- paste("PC", colnames(pca), sep = "")
pca<-as.data.frame(pca)
rownames(pca) <- sub(".bam$", "",rownames(pca))
pca <- pca[1:(nrow(pca) - 1), ]

pca<- tibble::rownames_to_column(pca)%>% dplyr::rename(sample=1)

# Assuming you have PCA results stored in a variable called pca_data
# Example data structure: pca_data <- prcomp(your_data)



# Plotting the PCA using ggplot2
library(ggplot2)

# Create a data frame for plotting

ggplot(pca, aes(x = PC1, y = PC2, color = sample)) +
  geom_point(size = 2) +
  labs(title = "PCA cfChromatin", x = "PC1", y = "PC2") +
  theme_classic(base_size=11) +
  theme(
    legend.position = "bottom",
    legend.key.size = unit(0.01, "cm"),  # Adjust the legend key size
    legend.text = element_text(size = 9),
    axis.text.y=element_text(size = 13),
    axis.text.x=element_text(size = 13),
    axis.title.x = element_text(size = 15),
    axis.title.y = element_text(size = 15),
    axis.title = element_text(size = 12))+  # Adjust the text size in the legend
  guides(color = guide_legend(title = "sample"))


ggsave("PCA.png", scale=0.5,plot = last_plot(), device = NULL, path = NULL, width =650, height = 315, units = "mm", dpi = 300, limitsize = TRUE)  



In [3]:
library(Rsamtools)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(GenomicAlignments)

Loading required package: Biostrings

Loading required package: XVector


Attaching package: ‘XVector’


The following object is masked from ‘package:purrr’:

    compact



Attaching package: ‘Biostrings’


The following object is masked from ‘package:base’:

    strsplit


Loading required package: GenomicFeatures

Loading required package: AnnotationDbi

Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.



Attaching package: ‘AnnotationDbi’


The following object is masked from ‘package:MASS’:

    select


The following object is masked from ‘package:plyranges’:

    select


The following object is masked from ‘package:dplyr’:

    select


Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘matrixStats’


In [ ]:
#' Transcription Start Site (TSS) Enrichment Score
#' @description TSS score is a raio between aggregate distribution of reads centered on TSSs and that flanking 
#' the corresponding TSSs. TSS score = the depth of TSS (each step within 1000 bp each side) / the depth of end flanks (100bp each end).
#' TSSE score = max(mean(TSS score in each window)).
#' @param obj an object of \link[GenomicAlignments:GAlignments-class]{GAlignments}
#' @param txs GRanges of transcripts
#' @param seqlev A vector of characters indicates the sequence levels.
#' @param upstream,downstream numeric(1) or integer(1). upstream and downstream of TSS. Default is 1000
#' @param endSize numeric(1) or integer(1). the size of the end flanks. Default is 100
#' @param width numeric(1) or integer(1). the window size for TSS score. Default is 100.
#' @param step numeric(1) or integer(1). The distance between the start position of the sliding windows.
#' @param pseudocount numeric(1) or integer(1). Pseudocount. Default is 0. 
#' If pseudocount is no greater than 0, the features with ZERO or less than ZERO 
#' counts in flank region will be removed in calculation. 
#' @importClassesFrom GenomicAlignments GAlignments
#' @importClassesFrom GenomicRanges GRanges
#' @importFrom GenomicRanges promoters coverage shift
#' @importFrom IRanges viewMeans Views
#' @export
#' @return A object of \link[GenomicRanges:GRanges-class]{GRanges} with TSS scores
#' @author Jianhong Ou
#' @references https://www.encodeproject.org/data-standards/terms/#enrichment
#' @examples  
#' library(GenomicRanges)
#' bamfile <- system.file("extdata", "GL1.bam", 
#'                        package="ATACseqQC", mustWork=TRUE)
#' gal1 <- readBamFile(bamFile=bamfile, tag=character(0), 
#'                     which=GRanges("chr1", IRanges(1, 1e6)), 
#'                     asMates=FALSE)
#' library(TxDb.Hsapiens.UCSC.hg19.knownGene)
#' txs <- transcripts(TxDb.Hsapiens.UCSC.hg19.knownGene)
#' tsse <- TSSEscore(gal1, txs)
#this is the function to calculate TSS enrichment 
TSSEscore <- function(obj, txs,
                      seqlev=intersect(seqlevels(obj), seqlevels(txs)),
                      upstream=1000, downstream=1000, endSize=100, 
                      width=100, step = width, pseudocount=0){
  stopifnot(is(obj, "GAlignments"))
  if(length(obj)==0){
    obj <- loadBamFile(obj, minimal=TRUE)
  }
  stopifnot(is(txs, "GRanges"))
  obj <- as(obj, "GRanges")
  mcols(obj) <- NULL
  cvg <- coverage(obj)
  cvg <- cvg[sapply(cvg, mean)>0]
  cvg <- cvg[names(cvg) %in% seqlev]
  seqlev <- seqlev[seqlev %in% names(cvg)]
  cvg <- cvg[seqlev]
  cvg <- cvg + pseudocount
  txs <- txs[seqnames(txs) %in% seqlev]
  txs <- unique(txs)
  sel.center <- promoters(txs, upstream = upstream, downstream = downstream)
  sel.center$id <- seq_along(sel.center)
  sel.center.sw <- slidingWindows(sel.center, width = width, step = step)
  names(sel.center.sw) <- sel.center$id
  sel.center.s <- unlist(sel.center.sw, use.names = TRUE)
  sel.center.s$idx <- unlist(lapply(lengths(sel.center.sw), seq.int))
  
  sel.center.s <- split(sel.center.s, seqnames(sel.center.s))
  sel.center.s <- sel.center.s[names(cvg)]
  
  vws.center <- Views(cvg, sel.center.s)
  vms.center <- viewMeans(vws.center)
  
  ## do norm  
  #sel.center <- promoters(txs, upstream = upstream - endSize,
  #                        downstream = downstream - endSize
  #)
  sel.left.flank <- flank(sel.center, width=endSize, both=FALSE)
  sel.right.flank <- flank(sel.center, width=endSize, start=FALSE, both = FALSE)
  names(sel.left.flank) <- sel.left.flank$id
  names(sel.right.flank) <- sel.right.flank$id
  
  sel.left.flank <- split(sel.left.flank, seqnames(sel.left.flank))
  sel.left.flank <- sel.left.flank[names(cvg)]
  sel.right.flank <- split(sel.right.flank, seqnames(sel.right.flank))
  sel.right.flank <- sel.right.flank[names(cvg)]
  
  vws.left <- Views(cvg, sel.left.flank)
  vms.left <- viewMeans(vws.left)
  vws.right <- Views(cvg, sel.right.flank)
  vms.right <- viewMeans(vws.right)
  
  vms.m <- mapply(vms.center, sel.center.s, vms.left, vms.right, 
                  FUN = function(v, i, vl, vr){
    i <- i$idx
    id <- sort(union(names(vl), names(vr)))##make sure the left and right paired
    vl <- vl[id]
    vr <- vr[id]
    vl[is.na(vl)] <- vr[is.na(vl)]
    vr[is.na(vr)] <- vl[is.na(vr)]
    vl[is.na(vl)] <- pseudocount
    vr[is.na(vr)] <- pseudocount
    blk <- vl+vr
    names(blk) <- id
    keep <- blk>0 ## in case pseudocount is less than 1
    blk <- blk[keep]
    blk <- blk/2
    keep <- names(v) %in% names(blk)
    v <- v[keep]
    i <- i[keep]
    v <- v*endSize/blk[names(v)]/width
    tt <- table(i)
    rs <- rowsum(v, i) ## possible error for MAX_FLOAT
    if(length(rs)!=length(tt)) return(NULL)
    tt <- tt[rownames(rs)]
    tt[is.na(tt)] <- max(tt, na.rm = TRUE)
    names(tt) <- rownames(rs)
    rs <- cbind(rs, tt)
  }, SIMPLIFY = FALSE)
  
  tt <- do.call(rbind, lapply(vms.m, function(.ele) .ele[, 2]))
  vms.m <- do.call(rbind, lapply(vms.m, function(.ele) .ele[, 1]))
  vms.m <- colSums(vms.m)/colSums(tt)
  
  TSSE <- max(vms.m, na.rm = TRUE)
  return(list(values=vms.m, TSSEscore=TSSE))
}

In [ ]:
library(GenomicAlignments)
library(GenomicFeatures)
library(Rsamtools)
library(parallel)

# Your directory path containing BAM files
directory_path <- "/date/gcb/gcb_MZ/Round1/filter/samtools"

# List the BAM files with specific naming conventions
bam_files <- list.files(directory_path, pattern = ".*H3K27ac.*\\.bam$", full.names = TRUE)

num_cores <- 20

# Function to process a single BAM file and save the plot
processBAM <- function(bam_file) {
  bam <- readGAlignments(bam_file)
  
  # Obtain transcripts data
  txs <- transcripts(TxDb.Hsapiens.UCSC.hg38.knownGene)
  
  seqlev <- unique(as.character(seqnames(txs)))
  seqinformation <- seqinfo(TxDb.Hsapiens.UCSC.hg38.knownGene)
  which <- as(seqinformation[seqlev], "GRanges")
  
  gal <- bam
  
  tsse <- TSSEscore(gal, txs)

  filename <- paste0(basename(bam_file), "_TSS.png")
  png(filename = filename, width = 10, height = 10, res = 720, units = "cm")
  plot(100 * (-9:10 - 0.5), tsse$values, type = "b",
       xlab = "Distance to TSS",
       ylab = "Aggregate TSS score")
  dev.off()

  return(tsse)
}

# Use parallel::mclapply to process BAM files in parallel and store results
results <- mclapply(bam_files, processBAM, mc.cores = num_cores)





In [4]:
#' @title fragment size distribution
#' @description estimate the fragment size of bams
#' @param bamFiles A vector of characters indicates the file names of bams.
#' @param index The names of the index file of the 'BAM' file being processed;
#'        This is given without the '.bai' extension.
#' @param bamFiles.labels labels of the bam files, used for pdf file naming.
#' @param ylim numeric(2). ylim of the histogram.
#' @param logYlim numeric(2). ylim of log-transformed histogram for the insert.
#' @return Invisible fragment length distribution list.
#' @importFrom Rsamtools ScanBamParam scanBamFlag scanBam idxstatsBam
#' @importFrom graphics axis par
#' @import GenomicRanges
#' @export
#' @author Jianhong Ou
#' @examples
#' bamFiles <- dir(system.file("extdata", package="ATACseqQC"), "GL.*.bam$", full.names=TRUE)
#' bamFiles.labels <- sub(".bam", "", basename(bamFiles))
#' fragSizeDist(bamFiles, bamFiles.labels)

fragSizeDist <- function(bamFiles, bamFiles.labels, index=bamFiles, ylim=NULL,
                         logYlim=NULL){
  opar <- par(c("fig", "mar"))
  on.exit(par(opar))
  pe <- mapply(testPairedEndBam, bamFiles, index)
  if(any(!pe)){
    stop(paste(bamFiles[!pe], collapse = ", "), 
         "is not Paired-End file.")
  }
  summaryFunction <- function(seqname, seqlength, bamFile, ind, ...) {
    param <-
      ScanBamParam(what=c('isize'),
                   which=GRanges(seqname, IRanges(1, seqlength)),
                   flag=scanBamFlag(isSecondaryAlignment = FALSE,
                                    isUnmappedQuery=FALSE,
                                    isNotPassingQualityControls = FALSE))
    table(abs(unlist(sapply(scanBam(bamFile, index=ind, ..., param=param), 
                            `[[`, "isize"), use.names = FALSE)))
  }

  idxstats <- unique(do.call(rbind, mapply(function(.ele, .ind)
    idxstatsBam(.ele, index = .ind)[, c("seqnames", "seqlength")], bamFiles, index, SIMPLIFY=FALSE)))
  seqnames <- as.character(idxstats[, "seqnames"])
  seqlen <- as.numeric(idxstats[, "seqlength"])
  fragment.len <- mapply(function(bamFile, ind) summaryFunction(seqname=seqnames, seqlength=seqlen, bamFile, ind), 
                         bamFiles, index, SIMPLIFY=FALSE)

  names(fragment.len) <- bamFiles.labels

  minor.ticks.axis <- function(ax,n=9,t.ratio=0.5,mn,mx,...){

    lims <- par("usr")
    lims <- if(ax %in% c(1,3)) lims[1:2] else lims[3:4]

    major.ticks <- pretty(lims,n=5)
    if(missing(mn)) mn <- min(major.ticks)
    if(missing(mx)) mx <- max(major.ticks)

    major.ticks <- major.ticks[major.ticks >= mn & major.ticks <= mx]

    labels <- sapply(major.ticks,function(i)
      as.expression(bquote(10^ .(i)))
    )
    axis(ax,at=major.ticks,labels=labels,
         las=ifelse(ax %in% c(2, 4), 2, 1), ...)

    n <- n+2
    minors <- log10(pretty(10^major.ticks[1:2],n))-major.ticks[1]
    minors <- minors[-c(1,n)]

    minor.ticks = c(outer(minors,major.ticks,`+`))
    minor.ticks <- minor.ticks[minor.ticks > mn & minor.ticks < mx]


    axis(ax,at=minor.ticks,tcl=par("tcl")*t.ratio,labels=FALSE)
  }

  null <- mapply(function(frag.len, frag.name){
    x <- 1:1010
    frag.len <- frag.len[match(x, names(frag.len))]
    frag.len[is.na(frag.len)] <- 0
    y <- frag.len / sum(frag.len)
    y <- as.numeric(y)
    names(y) <- x
    par(mar=c(5, 5, 4, 2) +.1)
    plot(x, y*10^3, main=paste(frag.name, "fragment sizes"),
         xlim=c(0, 1010), ylim=ylim,
         xlab="Fragment length (bp)",
         ylab=expression(Normalized ~ read ~ density ~ x ~ 10^-3),
         type="l")
    par(fig=c(.4, .95, .4, .95), new=TRUE)
    plot(x, log10(y), xlim=c(0, 1010), ylim=logYlim,
         xlab="Fragment length (bp)", ylab="Norm. read density",
         type="l", yaxt="n")
    minor.ticks.axis(2)
    par(opar)
  }, fragment.len, names(fragment.len))

  return(invisible(fragment.len))
}

In [15]:
library(GenomicAlignments)
library(GenomicFeatures)
library(Rsamtools)
library(parallel)

# Your directory path containing BAM files
directory_path <- "/date/gcb/gcb_MZ/dedup/picard"

# List the BAM files with specific naming conventions
bam_files <- list.files(directory_path, pattern = ".*H3K27ac.*\\.bam$", full.names = TRUE)

num_cores <- 20
# Function to process a single BAM file and save the plot
fragSIZE <- function(bam_file) {
## generate fragement size distribution
bamfile.labels <- gsub(".bam", "", basename(bamfile))
filename <- paste0(basename(bam_file), "_FragSize.png")
png(filename = filename, width = 10, height = 10, res = 720, units = "cm")
fragSize <- fragSizeDist(bamfile, bamfile.labels)
dev.off()
}

# Use parallel::mclapply to process BAM files in parallel and store results
results <- mclapply(bam_files,fragSIZE, mc.cores = num_cores)


In [16]:

fragSize <- fragSizeDist(bam_files,bamfile.labels)

Here counts in Hg38 for H3K27ac samples single end are computed 

In [ ]:
#Calculate correlation between TSS using Deseq2

library (DESeq2)
library(Rsubread)
#Bam files list where to perform Counts calculation

bamsToCount <- dir("/date/gcb/gcb_MZ/Round1/dedup/picard/SE/H3K27ac/", full.names = TRUE, pattern = "*.\\.bam$")
bamsToCount
# Calculate differential accessibility using DESEQ2


fcResults_SE <- featureCounts(bamsToCount, annot.ext = regionsToCount, isPairedEnd = FALSE,
                           countMultiMappingReads = FALSE, maxFragLength = 100,
                           nthreads = 20)


myCounts_SE <- fcResults$counts

myCounts_SE

The two matrices now will be bound together to build a unique matrix

In [ ]:
# Extract sample type and sample name
sample_type <-  sub(".*_(ChIP|Cut-Tag)\\s*$", "\\1", new_colnames)
sample_name <- new_colnames

# Create the sampleinfo data frame
sampleinfo <- data.frame(sampletype = sample_type, samplename = sample_name)

# Now, sampleinfo contains the desired table for DESeq2 analysis
sampleinfo

To compute the counts to perform PCA and the rest of the analysis will use Deseq

In [ ]:
myCounts_DF<- as.data.frame(myCounts)
Geneid<-rownames(myCounts_DF)

GeneID<-as.data.frame(Geneid)%>% 
dplyr::rename(GeneID=1)
GeneID

GeneID <- GeneID %>%
  separate(GeneID, into = c("chr", "start", "end"), sep = "_")

GeneID

consensusToCount2<-makeGRangesFromDataFrame(GeneID)

consensusToCount2

cfChrom_DDS <- DESeqDataSetFromMatrix(myCounts, metaData, design = ~Group, rowRanges = consensusToCount2)

cfChrom_DDS <- DESeq(cfChrom_DDS)


In [ ]:
regionsToCount